In [0]:
from pyspark.sql.functions import * 

In [0]:
spark.conf.set('fs.azure.account.key.byanythingstorageaccount.dfs.core.windows.net','Rmv4nXVTTn5udrwp7Lorm3fQgQyhrBzYf7R6OWfggdR5BxgjBEzSoXs7qQ7JVx/4zJY8MAMsNoQG+AStm1PdYw==')

# Load all dimensions

In [0]:
df_customer=spark.sql('''
 select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_customer`   
''')
df_customer.display()

CustomerID,CustomerName,Country,Dim_Customer_Key
9895.0,Matthew Stark,Libyan Arab Jamahiriya,1
2610.0,Krista Sampson,Jamaica,2
3565.0,James Rose,Zimbabwe,3
9555.0,Angela Pierce,Ethiopia,4
1349.0,Holly Jones,Solomon Islands,5
4664.0,Eric Olson,Belize,6
1792.0,Patrick Brown,French Southern Territories,7
8443.0,Joseph Wise,Jersey,8
7494.0,Christopher Leon,New Zealand,9
3226.0,Ryan Finley,Luxembourg,10


In [0]:
df_date=spark.sql('''
                  select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_date`
                  ''')
df_date.display()

OrderDate,Year,Month,Day,Dim_Date_Key
2024-10-24,2024,10,24,512
2024-10-24,2024,10,24,1
2024-08-20,2024,8,20,513
2024-08-20,2024,8,20,2
2024-07-14,2024,7,14,514
2024-07-14,2024,7,14,3
2024-09-15,2024,9,15,515
2024-09-15,2024,9,15,4
2024-08-06,2024,8,6,516
2024-08-06,2024,8,6,5


In [0]:
df_product=spark.sql('''
                     select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_product`
                     ''')
df_product.display()

ProductID,ProductCategory,ProductName,Dim_Product_Key
893.0,Beauty,Shampoo,1
730.0,Clothing,Jacket,2
642.0,Sports,Basketball,3
958.0,Electronics,Smartphone,4
892.0,Sports,Bicycle,5
664.0,Furniture,Chair,6
686.0,Sports,Treadmill,7
883.0,Sports,Basketball,8
613.0,Sports,Bicycle,9
761.0,Electronics,Headphones,10


In [0]:
df_region=spark.sql('''
                         select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_salesregion`
                         ''')
df_region.display()

SalesRegion,Dim_SalesRegion_Key
Central,1
South,2
East,3
West,4
North,5


# Load Silver Data

In [0]:
silver_df=spark.sql('''
                    select * from parquet.`abfss://siilver@byanythingstorageaccount.dfs.core.windows.net/`
                    ''')
silver_df.display()

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5861133349586922>, line 1
----> 1 silver_df=spark.sql('''
      2                     select * from parquet.`abfss://siilver@byanythingstorageaccount.dfs.core.windows.net/`
      3                     ''')
      4 silver_df.display()

File /databricks/spark/python/pyspark/sql/session.py:1943, in SparkSession.sql(self, sqlQuery, args, **kwargs)
   1938     else:
   1939         raise PySparkTypeError(
   1940             errorClass="INVALID_TYPE",
   1941             messageParameters={"arg_name": "args", "arg_type": type(args).__name__},
   1942         )
-> 1943     return DataFrame(self._jsparkSession.sql(sqlQuery, litArgs), self)
   1944 finally:
   1945     if len(kwargs) > 0:

File /databricks/spark/python/lib/py4j-0.10.9.9-src.zip/py4j/java_gateway.py:1362, in JavaMember.__call__(self, *args)
   1356 command = proto.

# Create fact table


In [0]:

fact_df = silver_df.join(df_customer, silver_df['CustomerID'] == df_customer['CustomerID'], 'left') \
    .join(df_product, silver_df['ProductID'] == df_product['ProductID'], 'left') \
    .join(df_date, silver_df['OrderDate'] == df_date['OrderDate'], 'left') \
    .join(df_region, silver_df['SalesRegion'] == df_region['SalesRegion'], 'left') \
    .select(
        df_customer['Dim_Customer_Key'],
        df_product['Dim_Product_Key'],
        df_date['Dim_Date_Key'],
        df_region['Dim_SalesRegion_Key'],
        silver_df['UnitPrice'],
        silver_df['TotalPrice'],
        silver_df['Quantity']
    )


fact_df.display()

In [0]:
from delta.tables import DeltaTable

path= 'abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_factsales'
# 1. PRE-PROCESS: Remove duplicates from your Fact DataFrame
# This ensures each combination of keys appears only once
fact_df_unique = fact_df.dropDuplicates([
    'Dim_Customer_Key', 
    'Dim_Product_Key', 
    'Dim_Date_Key', 
    'Dim_SalesRegion_Key'
])

# 2. Updated Merge Condition
merge_condition = """
    target.Dim_Customer_Key = source.Dim_Customer_Key AND 
    target.Dim_Product_Key = source.Dim_Product_Key AND 
    target.Dim_Date_Key = source.Dim_Date_Key AND 
    target.Dim_SalesRegion_Key = source.Dim_SalesRegion_Key
"""

# 3. Perform the Merge using the UNIQUE dataframe
if DeltaTable.isDeltaTable(spark, path):
    deltaTable = DeltaTable.forPath(spark, path)
    
    deltaTable.alias('target').merge(
        fact_df_unique.alias('source'), # Use the deduplicated DF here
        merge_condition
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
    
    print('Merge Operation Successful: dim_factsales updated.')
else:
    fact_df_unique.write.format('delta') \
        .mode('overwrite') \
        .option('mergeSchema', 'true') \
        .save(path)
    
    print('The table dim_factsales has been created.')

In [0]:
%sql
select * from delta.`abfss://gold@byanythingstorageaccount.dfs.core.windows.net/dim_factsales`